# OP26 — Phase 0 & 1: Preprocessing
**Agentic Dynamic Tariff Optimization for EV Charging Networks**

This notebook builds the *unified analytical base* for all downstream phases. The heavy lifting lives in `preprocess.py` (importable, tested); this notebook is the narrated, reproducible driver. Run top-to-bottom to regenerate every file in `outputs/`.

The two datasets are kept **separate by design** (different geography, role, and units) — see `ASSUMPTIONS.md`, decision #1.

## Phase 0 — setup
All paths, thresholds, and cleaning rules are centralised in `config.py`. Point the pipeline at the raw OpenProject files via `export OP26_DATA=/path/to/files` or by dropping them in `./data_raw/`.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))  # repo root: config.py + preprocess.py
import pandas as pd
import config as C
import preprocess as P
pd.set_option('display.max_columns', 50)
print('Reading raw data from:', C.DATA_DIR)

## Phase 1a — ACN (Caltech) session cleaning
Flatten the JSON-dump spreadsheet, drop wrapper-artifact/null rows, parse RFC-2822 timestamps to Caltech local time, and engineer session features: dwell (`duration_hr`), actual charging (`charging_hr`), post-charge idle/overstay (`idle_hr`), scheduling slack (`laxity_hr`), and `avg_power_kw`. Invalid charging times and implausible power are nulled **and flagged**, never silently dropped.

In [ ]:
acn, acn_report = P.clean_acn()
print(f"{len(acn):,} clean sessions | {acn.userID.nunique()} users | {acn.stationID.nunique()} stations")
print(f"with user input (enables laxity): {int(acn.has_user_input.sum()):,}")
acn[['sessionID','connect_local','duration_hr','charging_hr','idle_hr','kWhDelivered','avg_power_kw','laxity_hr','has_user_input']].head()

In [ ]:
acn[['duration_hr','charging_hr','idle_hr','laxity_hr','avg_power_kw','kWhDelivered','user_session_count']].describe().T

## Phase 1b — UrbanEV (Shenzhen) hourly panel
Aggregate the four 5-min matrices (8,640 steps) to **hourly** per zone (occupancy→mean, energy/charging/revenue→sum, price→mean), and engineer the brief's economic features: `utilization = clip(occupancy/piles, 0, 1)`, `revenue`, `revenue_per_kwh`, `occupancy_density`, and a **queue/saturation proxy** (`saturation_count`: 5-min intervals at ≥95% capacity). Calendar features, cyclical encodings, zone metadata (CBD, piles), and lag/rolling features are added per zone (time-ordered, leakage-safe).

In [ ]:
panel, zone_features, ev_report = P.build_urbanev_panel()
print(f"panel: {panel.shape[0]:,} rows ({panel.zone.nunique()} zones x {panel.hour_index.nunique()} hours), {panel.shape[1]} columns")
panel[['timestamp','zone','occupancy_mean','capacity','utilization','energy_kwh','price_mean','revenue','saturation_count','util_band']].head()

## Validation & reconciliation
Total energy in the hourly panel must equal the raw 5-min sum exactly, utilization must be in [0,1], and lag NaNs should be warm-up only.

In [ ]:
raw_energy = pd.read_csv(C.DATA_DIR / C.F_VOLUME).iloc[:,1:].values.sum()
print('energy reconciliation  raw =', round(raw_energy), ' panel =', round(panel.energy_kwh.sum()),
      ' -> match:', abs(raw_energy - panel.energy_kwh.sum()) < 1)
print('utilization range :', round(panel.utilization.min(),3), '..', round(panel.utilization.max(),3))
print('mean utilization  :', round(panel.utilization.mean(),3))
print('util-band share   :')
print(panel.util_band.value_counts(normalize=True).round(3).to_string())

**Key finding that shapes the pricing strategy:** the network is mostly idle — ~61% of zone-hours sit below 30% utilization and only ~1% above 80%. The dominant lever is **discount-driven off-peak uplift**, with targeted surge on the rare hot zones.

In [ ]:
# busiest zones by mean utilization (spatial profile for Phase 2 EDA)
zone_features.sort_values('util_mean', ascending=False)[
    ['zone','CBD','dynamic_pricing','capacity','util_mean','util_max','pct_hours_peak','pct_hours_offpeak']].head(8)

## Write outputs
Everything downstream reads from these files.

In [ ]:
acn.to_csv(C.OUTPUT_DIR / 'clean_acn_sessions.csv', index=False)
panel.to_csv(C.OUTPUT_DIR / 'urbanev_panel_hourly.csv.gz', index=False, compression='gzip')
panel.head(2000).to_csv(C.OUTPUT_DIR / 'urbanev_panel_hourly_sample.csv', index=False)
zone_features.to_csv(C.OUTPUT_DIR / 'zone_features.csv', index=False)
qa = {**acn_report, **ev_report}
pd.DataFrame([{'metric':k,'value':v} for k,v in qa.items()]).to_csv(C.OUTPUT_DIR / 'data_quality_report.csv', index=False)
print('Wrote outputs to', C.OUTPUT_DIR)
for f in sorted(C.OUTPUT_DIR.glob('*')): print('  ', f.name)

---
**Phase 1 complete.** Next: Phase 2 (EDA + empirical peak-window definition) reads `outputs/urbanev_panel_hourly.csv.gz` and `outputs/clean_acn_sessions.csv`.